<a href="https://colab.research.google.com/github/rakeshma-72/LLM/blob/main/Experiment_9_Retrieval_Augmented_Generation_(RAG)_with_FAISS_Vector_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

RAKESH M A - ENG23CS0157

Experiment 9 -Retrieval-Augmented Generation (RAG) with FAISS Vector Search

In [1]:
!pip install -q torch transformers accelerate sentence-transformers faiss-cpu

In [2]:
import glob
import os
import torch
import faiss
import pandas as pd

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

DEV = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEV)

tok = AutoTokenizer.from_pretrained(
    MODEL,
    padding_side="left"
)

if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16 if DEV == "cuda" else torch.float32
).to(DEV).eval()

print("Model loaded successfully!")

Device: cuda


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [3]:
@torch.no_grad()
def generate(
    prompt,
    system="You are a helpful, concise assistant.",
    max_new_tokens=200
):

    text = tok.apply_chat_template(
        [
            {"role": "system", "content": system},
            {"role": "user", "content": prompt}
        ],
        tokenize=False,
        add_generation_prompt=True
    )

    enc = tok(text, return_tensors="pt").to(DEV)

    out = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tok.pad_token_id
    )

    return tok.decode(
        out[0, enc["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

In [5]:
os.makedirs("docs", exist_ok=True)

SAMPLE = {

    "dsu.txt": (
        "Dayananda Sagar University is located at Devarakaggalahalli, "
        "Harohalli, Kanakapura Road, Bengaluru South District, Karnataka. "
        "Its School of Engineering offers the course Large Language "
        "Models with course code 23CS4701 in semester 7. "
        "The course carries 4 credits."
    ),

    "lora.txt": (
        "LoRA (Low-Rank Adaptation) freezes the pretrained weights "
        "and injects small trainable low-rank matrices into attention "
        "layers. This reduces trainable parameters by more than 99 "
        "percent and makes fine-tuning possible on a single GPU."
    ),

    "rag.txt": (
        "Retrieval-Augmented Generation combines a retriever with "
        "a generator. Documents are split into chunks, embedded, "
        "stored in a vector database such as FAISS or Chroma, "
        "and the top-k similar chunks are inserted in the prompt "
        "so the model answers from evidence and hallucinates less."
    ),

    "decoding.txt": (
        "Temperature rescales logits before sampling; lower values "
        "make output more deterministic. Top-k sampling keeps only "
        "the k most likely tokens, while top-p keeps the smallest "
        "set of tokens whose cumulative probability exceeds p."
    )
}

for name, content in SAMPLE.items():

    with open(f"docs/{name}", "w", encoding="utf-8") as f:
        f.write(content)

print("Documents created:", len(SAMPLE))

Documents created: 4


In [6]:
EVAL = [

    (
        "What is the course code of the Large Language Models course?",
        "dsu.txt",
        "23cs4701"
    ),

    (
        "How does LoRA reduce the cost of fine-tuning?",
        "lora.txt",
        "low-rank"
    ),

    (
        "Which vector databases are mentioned for RAG?",
        "rag.txt",
        "faiss"
    ),

    (
        "What does top-p sampling do?",
        "decoding.txt",
        "cumulative"
    )

]

print("Evaluation questions:", len(EVAL))

Evaluation questions: 4


In [7]:
def load_docs():

    files = glob.glob("docs/*.txt")

    return {
        os.path.basename(f):
        open(f, encoding="utf-8").read()
        for f in files
    }

documents = load_docs()

print("Loaded documents:", len(documents))

for name in documents:
    print(name)

Loaded documents: 4
rag.txt
dsu.txt
decoding.txt
lora.txt


In [8]:
def chunk(text, size=300, overlap=50):

    out = []
    i = 0

    while i < len(text):

        out.append(text[i:i + size])

        i += size - overlap

    return out


chunks = []
src = []

for name, text in documents.items():

    for c in chunk(text):

        chunks.append(c)
        src.append(name)

print("Total chunks:", len(chunks))

Total chunks: 6


In [9]:
emb = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

vecs = emb.encode(
    chunks,
    normalize_embeddings=True
).astype("float32")

print("Embedding shape:", vecs.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (6, 384)


In [10]:
index = faiss.IndexFlatIP(
    vecs.shape[1]
)

index.add(vecs)

print(
    f"Indexed {len(chunks)} chunks "
    f"from {len(set(src))} documents"
)

Indexed 6 chunks from 4 documents


In [11]:
def retrieve(q, k=3):

    qv = emb.encode(
        [q],
        normalize_embeddings=True
    ).astype("float32")

    scores, idx = index.search(qv, k)

    return [
        (chunks[i], src[i], float(s))
        for i, s in zip(idx[0], scores[0])
        if i >= 0
    ]

In [12]:
def rag_answer(q, k=3):

    hits = retrieve(q, k)

    ctx = "\n".join(
        f"[{n}] {c}"
        for c, n, _ in hits
    )

    prompt = (
        "Answer the question using ONLY the context. "
        "If the answer is not in the context, "
        "say 'Not found in documents.'\n\n"

        f"Context:\n{ctx}\n\n"

        f"Question: {q}"
    )

    return generate(prompt), hits

In [13]:
hit = 0
kw_rag = 0
kw_plain = 0

for q, s, kw in EVAL:

    ans, hits = rag_answer(q)

    plain = generate(q)

    ok = s in [h[1] for h in hits]

    hit += ok

    kw_rag += kw in ans.lower()

    kw_plain += kw in plain.lower()

    print(f"\nQ: {q}")

    print(
        "Retrieved:",
        [(h[1], round(h[2], 2)) for h in hits]
    )

    print(f"RAG: {ans[:200]!r}")

    print(f"No-RAG: {plain[:200]!r}")


Q: What is the course code of the Large Language Models course?
Retrieved: [('dsu.txt', 0.39), ('rag.txt', 0.26), ('dsu.txt', 0.26)]
RAG: 'The course code for the Large Language Models course is 23CS4701.'
No-RAG: 'The course code for the Large Language Models (LLM) specialization at Stanford University is LLM-101.'

Q: How does LoRA reduce the cost of fine-tuning?
Retrieved: [('lora.txt', 0.69), ('decoding.txt', 0.19), ('dsu.txt', 0.11)]
RAG: 'LoRA reduces the cost of fine-tuning by freezing the pretrained weights and injecting small trainable low-rank matrices into attention layers, which leads to more than 99% reduction in trainable param'
No-RAG: 'LoRA (Low-Rank Approximation) is a technique used in machine learning to reduce the computational complexity and storage requirements of large-scale models by approximating them with low-rank matrices'

Q: Which vector databases are mentioned for RAG?
Retrieved: [('rag.txt', 0.35), ('dsu.txt', 0.23), ('lora.txt', 0.19)]
RAG: 'For RAG, th

In [14]:
n = len(EVAL)

print("\n===== FINAL RESULTS =====")

print(
    f"Retrieval hit@3 = {hit/n:.2f}"
)

print(
    f"Keyword accuracy RAG = {kw_rag/n:.2f}"
)

print(
    f"Keyword accuracy plain LLM = {kw_plain/n:.2f}"
)


===== FINAL RESULTS =====
Retrieval hit@3 = 1.00
Keyword accuracy RAG = 1.00
Keyword accuracy plain LLM = 0.25


In [15]:
results = pd.DataFrame({

    "Metric": [
        "Retrieval Hit@3",
        "Keyword Accuracy RAG",
        "Keyword Accuracy Plain LLM"
    ],

    "Score": [
        hit/n,
        kw_rag/n,
        kw_plain/n
    ]

})

display(results)

results.to_csv(
    "exp9_results.csv",
    index=False
)

print("Results saved successfully!")

,Metric,Score
0,Retrieval Hit@3,1.00
1,Keyword Accuracy RAG,1.00
2,Keyword Accuracy Plain LLM,0.25


Results saved successfully!


In [16]:
from google.colab import files

files.download("exp9_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>